# 03· Modelo econométrico: AR(1)-GARCH(1,1) con errores t de Student

Este notebook estima el modelo base del proyecto y genera sus pronósticos de varianza a un paso en los mismos folds de validación que usará el Random Forest.

**Contenido**
1. Carga de datos y folds
2. Especificación del modelo
3. Estimación en el tramo de diagnóstico
4. Verificación de supuestos
5. Comparación con especificaciones alternativas
6. Pronósticos a un paso en los folds (Time Series Cross-Validation)
7. Errores por fold y estabilidad de parámetros
8. Evaluación en el test final
9. Guardar resultados
10. Conclusiones

## 0. Configuración

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from arch import arch_model

# Raíz del proyecto (funciona desde la raíz o desde notebooks/)
RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))
from src import config as cfg

PROC = RAIZ / "data" / "processed"
RES_DIR = RAIZ / "results"
FIG_DIR = RAIZ / "figures"
RES_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


### Configuración de parámetros de visualización
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.3})
warnings.filterwarnings("ignore", category=FutureWarning)


## 1. Carga de datos y folds
Se cargan los resultados de `01_preparacion_datos`:
- `wti_retornos.csv`: retorno logarítmico diario del WTI en % (`WTI`) y estimador de Parkinson en %² (`WTI_park`), sin las sesiones del 20 y 21 de abril de 2020.
- `folds.csv`: los 10 folds de Time Series Cross-Validation (2 repeticiones × 5 folds, ventana expansiva).

In [4]:
PROC

WindowsPath('c:/Users/ASUS/Desktop/Maestria IA USFQ/AprendizajeSupervisado/Proyectofinal_ml_wti/data/processed')

In [5]:
base_ret = pd.read_csv(PROC / "train_validation.csv", index_col=0, parse_dates=True)

In [6]:
FOLDS = pd.read_csv(PROC / "folds.csv",
                    parse_dates=["train_ini", "train_fin", "val_ini", "val_fin"])

In [7]:
base_ret

,WTI,WTI_park
Date,,
2010-01-05,0.318465,0.599044
2010-01-06,1.709655,3.807394
2010-01-07,-0.627108,0.636423
2010-01-08,0.108816,1.473154
2010-01-11,-0.278337,2.075756
...,...,...
2024-12-24,1.234407,0.845292
2024-12-26,-0.687085,1.482612
2024-12-27,1.397820,1.259815


In [2]:
# Especificación del modelo base (se usa igual en todo el notebook)
ESPEC = dict(mean="AR", lags=1, vol="GARCH", p=1, q=1, dist="t")

In [8]:
r = base_ret["WTI"].dropna()            # retorno en %
park = base_ret["WTI_park"]             # proxy de varianza (Parkinson) en %²

In [9]:
r

Date
2010-01-05    0.318465
2010-01-06    1.709655
2010-01-07   -0.627108
2010-01-08    0.108816
2010-01-11   -0.278337
                ...   
2024-12-24    1.234407
2024-12-26   -0.687085
2024-12-27    1.397820
2024-12-30    0.550887
2024-12-31    1.023067
Name: WTI, Length: 3769, dtype: float64

In [10]:
park

Date
2010-01-05    0.599044
2010-01-06    3.807394
2010-01-07    0.636423
2010-01-08    1.473154
2010-01-11    2.075756
                ...   
2024-12-24    0.845292
2024-12-26    1.482612
2024-12-27    1.259815
2024-12-30    1.490422
2024-12-31    0.934503
Name: WTI_park, Length: 3769, dtype: float64

In [11]:
EDA_FIN = FOLDS["train_fin"].min()      # fin del tramo de diagnóstico (train más corto)
EDA_FIN

Timestamp('2019-06-30 00:00:00')

In [12]:
print(f"Retornos: {len(r)} obs | {r.index.min().date()} → {r.index.max().date()}")
print(f"Tramo de diagnóstico: hasta {EDA_FIN.date()}")
display(FOLDS)

Retornos: 3769 obs | 2010-01-05 → 2024-12-31
Tramo de diagnóstico: hasta 2019-06-30


,rep,fold,train_ini,train_fin,val_ini,val_fin
0,1,1,2010-01-01,2019-12-31,2020-01-01,2020-12-31
1,1,2,2010-01-01,2020-12-31,2021-01-01,2021-12-31
2,1,3,2010-01-01,2021-12-31,2022-01-01,2022-12-31
3,1,4,2010-01-01,2022-12-31,2023-01-01,2023-12-31
4,1,5,2010-01-01,2023-12-31,2024-01-01,2024-12-31
5,2,1,2010-01-01,2019-06-30,2019-07-01,2020-06-30
6,2,2,2010-01-01,2020-06-30,2020-07-01,2021-06-30
7,2,3,2010-01-01,2021-06-30,2021-07-01,2022-06-30
8,2,4,2010-01-01,2022-06-30,2022-07-01,2023-06-30
9,2,5,2010-01-01,2023-06-30,2023-07-01,2024-06-30


## 2. Especificación del modelo
El modelo se aplica directamente sobre los retornos $r_t$ y estima conjuntamente, por máxima verosimilitud, una ecuación para la media y otra para la varianza:

**Media, AR(1):**
$$r_t = \mu + \phi\, r_{t-1} + \varepsilon_t$$

**Varianza, GARCH(1,1):**
$$\varepsilon_t = \sigma_t z_t, \qquad \sigma_t^2 = \omega + \alpha\, \varepsilon_{t-1}^2 + \beta\, \sigma_{t-1}^2$$

**Distribución:** $z_t \sim t_\nu$ estandarizada (media 0 y varianza 1).

**Justificación (EDA, tramo 2010 – jun 2019):**
- Los retornos son estacionarios (ADF, KPSS y PP), por lo que se cumple el supuesto previo de estacionariedad.
- La PACF de $r_t$ muestra una autocorrelación pequeña pero significativa en el rezago 1 → término AR(1) en la media.
- La ACF de $r_t^2$ y la prueba ARCH-LM muestran efectos ARCH muy significativos → componente GARCH en la varianza.
- Los retornos tienen exceso de curtosis y Jarque-Bera rechaza normalidad → errores t de Student.

**Pronóstico que se compara con el Random Forest:** la varianza condicional a un paso,
$$\hat\sigma^2_{t+1} = \hat\omega + \hat\alpha\, \hat\varepsilon_t^2 + \hat\beta\, \hat\sigma_t^2$$
que se evalúa contra el proxy de Parkinson del mismo día, $\sigma^2_{P,t+1}$.

## 3. Estimación en el tramo de diagnóstico (2010 – jun 2019)

In [ ]:
r_diag = r.loc[:EDA_FIN]
modelo = arch_model(r_diag, **ESPEC)
res = modelo.fit(disp="off")
print(res.summary())

In [ ]:
p = res.params
pv = res.pvalues
# Nombre del coeficiente AR(1): arch lo nombra como "<serie>[1]" (por ejemplo "WTI[1]")
PHI = [k for k in p.index if k.endswith("[1]") and not k.startswith(("alpha", "beta", "gamma"))][0]

tabla_param = pd.DataFrame({"estimación": p, "error estándar": res.std_err,
                            "estadístico t": res.tvalues, "p-valor": pv})
tabla_param.index = tabla_param.index.map({"Const": "μ (constante)", PHI: "φ (AR(1))",
                                           "omega": "ω", "alpha[1]": "α", "beta[1]": "β",
                                           "nu": "ν (grados de libertad)"}.get)
display(tabla_param.round(4))
print(f"Log-verosimilitud: {res.loglikelihood:.2f} | AIC: {res.aic:.2f} | BIC: {res.bic:.2f}")

_Interpretación (completar con los valores obtenidos):_
- **φ:** …
- **α:** reacción de la volatilidad a los choques del día anterior …
- **β:** persistencia de la volatilidad …
- **ν:** …

## 4. Verificación de supuestos
Si el modelo está bien especificado, los residuos estandarizados $z_t = \hat\varepsilon_t / \hat\sigma_t$ deben comportarse como ruido independiente con la distribución supuesta.

### 4.1 Restricciones de los parámetros y persistencia
- **Varianza positiva:** $\omega > 0$, $\alpha \ge 0$, $\beta \ge 0$.
- **Estacionariedad de la varianza:** $\alpha + \beta < 1$. Si la suma es cercana a 1, la volatilidad es muy persistente.
- **Estacionariedad de la media:** $|\phi| < 1$.
- **Distribución:** $\nu > 2$ para que la varianza sea finita y $\nu > 4$ para que la curtosis sea finita.

In [ ]:
persist = p["alpha[1]"] + p["beta[1]"]
supuestos = pd.DataFrame({
    "valor": [p["omega"], p["alpha[1]"], p["beta[1]"], persist, abs(p[PHI]), p["nu"]],
    "condición": ["> 0", ">= 0", ">= 0", "< 1", "< 1", "> 4"],
    "¿cumple?": [p["omega"] > 0, p["alpha[1]"] >= 0, p["beta[1]"] >= 0,
                 persist < 1, abs(p[PHI]) < 1, p["nu"] > 4],
}, index=["ω", "α", "β", "α + β (persistencia)", "|φ|", "ν"])
display(supuestos.round(4))

if persist < 1:
    var_largo_plazo = p["omega"] / (1 - persist)
    print(f"Vida media de un choque de volatilidad: {np.log(0.5) / np.log(persist):.1f} días")
    print(f"Varianza de largo plazo: {var_largo_plazo:.3f} %²  "
          f"(volatilidad diaria ≈ {np.sqrt(var_largo_plazo):.2f} %, "
          f"anualizada ≈ {np.sqrt(252 * var_largo_plazo):.1f} %)")
print(f"Volatilidad diaria muestral del tramo: {r_diag.std():.2f} %")

### 4.2 Significancia de los parámetros
$H_0$: el parámetro es igual a 0. Se espera rechazarla (p < 0,05) al menos para $\alpha$ y $\beta$.

In [ ]:
display(pd.DataFrame({"p-valor": pv, "significativo (5%)": pv < 0.05}).round(4))

### 4.3 Autocorrelación y efectos ARCH remanentes
- **Ljung-Box sobre $z_t$** — $H_0$: no hay autocorrelación en los residuos estandarizados (la media quedó bien modelada).
- **Ljung-Box sobre $z_t^2$** y **ARCH-LM sobre $z_t$** — $H_0$: no quedan efectos ARCH (la varianza quedó bien modelada).

Se busca **no rechazar** $H_0$ (p > 0,05).

In [ ]:
z = res.std_resid.dropna()

lb = pd.concat({"z_t": acorr_ljungbox(z, lags=[5, 10, 21]),
                "z_t²": acorr_ljungbox(z**2, lags=[5, 10, 21])}, names=["serie", "rezago"])
display(lb.round(4))

lm_stat, lm_p, f_stat, f_p = het_arch(z, nlags=10)
print(f"ARCH-LM sobre z_t (10 rezagos): estadístico = {lm_stat:.2f}, p = {lm_p:.4f}")

# Comparación con los retornos originales (antes del modelo)
lm_r = het_arch(r_diag - r_diag.mean(), nlags=10)
print(f"ARCH-LM sobre r_t (antes del modelo): estadístico = {lm_r[0]:.2f}, p = {lm_r[1]:.4g}")

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(z, lags=40, zero=False, ax=ax[0], title="ACF de los residuos estandarizados z_t")
plot_acf(z**2, lags=40, zero=False, ax=ax[1], title="ACF de z_t²")
for a in ax:
    a.set_ylim(-0.15, 0.15)
plt.tight_layout(); plt.savefig(FIG_DIR / "03a_acf_residuos.png", dpi=150); plt.show()

### 4.4 Distribución de los residuos estandarizados
Se compara $z_t$ con la distribución t estandarizada con los grados de libertad estimados $\hat\nu$ y con la normal. Si la t es adecuada, los puntos del primer QQ-plot deben seguir la diagonal, incluidas las colas.

In [ ]:
nu = p["nu"]
t_std = stats.t(df=nu, scale=np.sqrt((nu - 2) / nu))   # t con varianza 1, como en arch

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
stats.probplot(z, dist=t_std, plot=ax[0]); ax[0].set_title(f"QQ-plot de z_t vs t estandarizada (ν = {nu:.1f})")
stats.probplot(z, dist="norm", plot=ax[1]); ax[1].set_title("QQ-plot de z_t vs normal")
plt.tight_layout(); plt.savefig(FIG_DIR / "03a_qqplot_residuos.png", dpi=150); plt.show()

print(f"Asimetría de z_t: {stats.skew(z):.3f} | curtosis en exceso: {stats.kurtosis(z):.3f}")
print(f"Curtosis en exceso teórica de la t(ν): {6 / (nu - 4):.3f}" if nu > 4 else "ν <= 4: curtosis no finita")

### 4.5 Asimetría: sign bias test (Engle y Ng)
Evalúa si los choques negativos y positivos afectan de forma distinta a la volatilidad (efecto apalancamiento), algo que un GARCH simétrico no captura. Se estima por MCO:
$$z_t^2 = c_0 + c_1 S^-_{t-1} + c_2 S^-_{t-1} z_{t-1} + c_3 S^+_{t-1} z_{t-1} + u_t$$
donde $S^-_{t-1} = 1$ si $z_{t-1} < 0$ y $S^+_{t-1} = 1 - S^-_{t-1}$.

$H_0$: $c_1 = c_2 = c_3 = 0$ (no hay sesgo de signo; el GARCH simétrico es adecuado).

In [ ]:
neg = (z.shift(1) < 0).astype(float)
X_sb = pd.DataFrame({"S_neg": neg,
                     "S_neg_z": neg * z.shift(1),
                     "S_pos_z": (1 - neg) * z.shift(1)}).dropna()
ols_sb = sm.OLS((z**2).loc[X_sb.index], sm.add_constant(X_sb)).fit()
display(ols_sb.summary2().tables[1].round(4))
print(f"Prueba conjunta (F): estadístico = {ols_sb.fvalue:.3f}, p = {ols_sb.f_pvalue:.4f}")

### 4.6 Volatilidad condicional estimada
Se compara la volatilidad condicional del modelo, $\hat\sigma_t$, con la raíz del estimador de Parkinson, $\sqrt{\sigma^2_{P,t}}$, en el tramo de diagnóstico.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
np.sqrt(park.loc[r_diag.index]).plot(ax=ax, lw=0.5, alpha=0.5, color="grey", label="√Parkinson (%)")
res.conditional_volatility.plot(ax=ax, lw=1.2, color="tab:red", label="σ̂_t AR(1)-GARCH(1,1)-t (%)")
ax.set_title("Volatilidad condicional estimada vs Parkinson (2010 – jun 2019)")
ax.legend(); plt.tight_layout(); plt.savefig(FIG_DIR / "03a_volatilidad_condicional.png", dpi=150); plt.show()

### 4.7 Resumen de supuestos
_Completar con los resultados:_

| Supuesto | Prueba | Resultado | ¿Cumple? |
|---|---|---|---|
| Estacionariedad de $r_t$ | ADF, KPSS, PP (EDA) | … | ✅ |
| Efectos ARCH en $r_t$ | ARCH-LM (EDA) | … | ✅ |
| Varianza positiva | $\omega, \alpha, \beta \ge 0$ | … | |
| Varianza estacionaria | $\alpha + \beta < 1$ | … | |
| Sin autocorrelación en $z_t$ | Ljung-Box | … | |
| Sin efectos ARCH remanentes | Ljung-Box $z_t^2$, ARCH-LM | … | |
| Distribución t adecuada | QQ-plot, $\nu > 4$ | … | |
| Simetría | Sign bias test | … | |

## 5. Comparación con especificaciones alternativas
Para respaldar la especificación elegida se comparan, en el mismo tramo de diagnóstico, variantes del modelo con los criterios de información AIC y BIC (menor es mejor):
- **AR(0)-GARCH(1,1)-t:** sin término autorregresivo en la media.
- **AR(1)-GARCH(1,1)-normal:** errores normales en lugar de t.
- **AR(1)-GJR-GARCH(1,1)-t:** agrega un término de asimetría (los choques negativos pueden tener un efecto distinto).

In [ ]:
alternativas = {
    "AR(1)-GARCH(1,1)-t (elegido)": ESPEC,
    "AR(0)-GARCH(1,1)-t":           dict(ESPEC, mean="Constant", lags=0),
    "AR(1)-GARCH(1,1)-normal":      dict(ESPEC, dist="normal"),
    "AR(1)-GJR-GARCH(1,1)-t":       dict(ESPEC, o=1),
}
filas = []
for nombre, espec in alternativas.items():
    rr = arch_model(r_diag, **espec).fit(disp="off")
    pp_ = rr.params
    filas.append({"modelo": nombre, "n_param": len(pp_), "log-veros.": rr.loglikelihood,
                  "AIC": rr.aic, "BIC": rr.bic,
                  "persistencia": pp_["alpha[1]"] + pp_["beta[1]"] + 0.5 * pp_.get("gamma[1]", 0)})
comp_espec = pd.DataFrame(filas).set_index("modelo")
display(comp_espec.round(3).sort_values("BIC"))

_Comentario:_ si el modelo elegido tiene el menor BIC (o una diferencia pequeña frente al mejor), se mantiene. Si el GJR-GARCH mejora claramente y el sign bias test rechazó $H_0$, se menciona como alternativa o limitación.

## 6. Pronósticos a un paso en los folds (Time Series Cross-Validation)
En cada uno de los 10 folds:
1. Los parámetros se estiman **solo con el entrenamiento** del fold (desde 2010 hasta `train_fin`).
2. Con esos parámetros fijos, el modelo se filtra día a día sobre la validación. Para cada día objetivo $\tau$ se usa la información disponible hasta $\tau - 1$ para pronosticar $\hat\sigma^2_\tau$.
3. El pronóstico se compara con el proxy de Parkinson del mismo día, $\sigma^2_{P,\tau}$.

**Convención de índices:** los pronósticos se indexan por el **día objetivo** $\tau$ (el día que se pronostica). Así, un fold con validación de enero a diciembre de 2020 evalúa exactamente los días de 2020, y ningún día del test final entra en la validación. El Random Forest usa la misma convención.

**Funciones de pérdida:**
- $\text{MSE} = \frac{1}{n}\sum (\sigma^2_{P,\tau} - \hat\sigma^2_\tau)^2$
- $\text{QLIKE} = \frac{1}{n}\sum \left(\frac{\sigma^2_{P,\tau}}{\hat\sigma^2_\tau} + \ln \hat\sigma^2_\tau\right)$. Es robusta al ruido del proxy (Patton, 2011) y está definida aunque el proxy sea 0.

In [ ]:
def pronostico_garch(r, train_fin, obj_ini, obj_fin, espec=ESPEC):
    # Estima con datos hasta train_fin y pronostica la varianza a un paso
    # para cada día objetivo en [obj_ini, obj_fin].
    # Devuelve (Serie indexada por día objetivo, parámetros estimados).

    # 1. Estimación solo con el entrenamiento
    res_tr = arch_model(r.loc[:train_fin], **espec).fit(disp="off")

    # 2. Filtrado con parámetros fijos hasta el último día objetivo
    r_full = r.loc[:obj_fin]
    pos_ini = r_full.index.searchsorted(pd.Timestamp(obj_ini))   # primer día objetivo
    origen = pos_ini - 1                                          # se pronostica desde el día anterior
    fc = arch_model(r_full, **espec).forecast(params=res_tr.params, horizon=1,
                                              start=origen, reindex=False)
    h = fc.variance["h.1"].to_numpy()

    # La fila del origen t es el pronóstico para t+1: se reindexa al día objetivo
    objetivos = r_full.index[origen + 1:]
    h = pd.Series(h[:len(objetivos)], index=objetivos, name="h_garch")
    return h.loc[obj_ini:obj_fin], res_tr


def perdidas(y, h):
    return pd.DataFrame({"mse": (y - h)**2, "qlike": y / h + np.log(h)})

In [ ]:
filas, pron, params_folds = [], [], []

for f in FOLDS.itertuples():
    h, res_f = pronostico_garch(r, f.train_fin, f.val_ini, f.val_fin)
    df = pd.concat({"y": park, "h_garch": h}, axis=1).loc[h.index].dropna()
    L = perdidas(df["y"], df["h_garch"])

    filas.append({"rep": f.rep, "fold": f.fold,
                  "val_ini": f.val_ini.date(), "val_fin": f.val_fin.date(), "n": len(df),
                  "mse_garch": L["mse"].mean(), "qlike_garch": L["qlike"].mean(),
                  "convergencia": res_f.convergence_flag == 0})
    pron.append(df.assign(rep=f.rep, fold=f.fold))
    params_folds.append(res_f.params.rename(f"R{f.rep}·F{f.fold}"))
    print(f"Rep {f.rep} · Fold {f.fold}: {len(df)} días | QLIKE = {L['qlike'].mean():.4f}")

res_garch = pd.DataFrame(filas)
pron_garch = pd.concat(pron)
params_garch = pd.concat(params_folds, axis=1).T

## 7. Errores por fold y estabilidad de parámetros

In [ ]:
display(res_garch.round(4))
print("Promedio sobre los 10 folds:")
display(res_garch[["mse_garch", "qlike_garch"]].agg(["mean", "std"]).round(4))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
etiq = [f"R{a}·F{b}" for a, b in zip(res_garch["rep"], res_garch["fold"])]
ax[0].bar(etiq, res_garch["mse_garch"]); ax[0].set_title("MSE por fold"); ax[0].tick_params(axis="x", rotation=45)
ax[1].bar(etiq, res_garch["qlike_garch"], color="tab:orange"); ax[1].set_title("QLIKE por fold"); ax[1].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.savefig(FIG_DIR / "03a_errores_por_fold.png", dpi=150); plt.show()

In [ ]:
# Estabilidad de los parámetros a medida que el entrenamiento se expande
tabla_est = params_garch.rename(columns={PHI: "φ", "Const": "μ", "omega": "ω",
                                         "alpha[1]": "α", "beta[1]": "β", "nu": "ν"})
tabla_est["α + β"] = tabla_est["α"] + tabla_est["β"]
display(tabla_est.round(4))

fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
for a, c in zip(ax, ["α", "β", "ν"]):
    a.plot(tabla_est.index, tabla_est[c], "o-"); a.set_title(c); a.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.savefig(FIG_DIR / "03a_estabilidad_parametros.png", dpi=150); plt.show()

In [ ]:
# Pronóstico vs Parkinson en dos folds representativos
fig, axes = plt.subplots(2, 1, figsize=(13, 7))
for ax, (rep, fold) in zip(axes, [(1, 1), (1, 3)]):
    d = pron_garch.query("rep == @rep and fold == @fold")
    ax.plot(d.index, d["y"], lw=0.6, color="grey", alpha=0.7, label="Parkinson")
    ax.plot(d.index, d["h_garch"], lw=1.3, color="tab:red", label="Pronóstico AR(1)-GARCH(1,1)-t")
    ax.set_title(f"Rep {rep} · Fold {fold}: validación {d.index.min().date()} → {d.index.max().date()}")
    ax.set_ylabel("%²"); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "03a_pronosticos_folds.png", dpi=150); plt.show()

## 8. Evaluación en el test final (2025 – oct 2026)
Se estima el modelo con todo el periodo de desarrollo (2010–2024) y se pronostica cada día del test final.

> **Ejecutar solo al terminar el proyecto**, cuando el Random Forest también esté definido, para que ningún modelo se ajuste mirando el test. Cambiar `EVALUAR_TEST` a `True`.

In [ ]:
EVALUAR_TEST = False

if EVALUAR_TEST:
    h_test, res_dev = pronostico_garch(r, cfg.DEV_FIN, cfg.TEST_INI, cfg.TEST_FIN)
    test_garch = pd.concat({"y": park, "h_garch": h_test}, axis=1).loc[h_test.index].dropna()
    L_test = perdidas(test_garch["y"], test_garch["h_garch"])
    print(f"Test final: {len(test_garch)} días | MSE = {L_test['mse'].mean():.4f} | QLIKE = {L_test['qlike'].mean():.4f}")
    display(res_dev.params.round(4).to_frame("parámetros (desarrollo completo)"))
    test_garch.to_csv(RES_DIR / "garch_pronosticos_test.csv")

## 9. Guardar resultados

In [ ]:
res_garch.to_csv(RES_DIR / "garch_errores_por_fold.csv", index=False)
pron_garch.to_csv(RES_DIR / "garch_pronosticos_diarios.csv")
params_garch.to_csv(RES_DIR / "garch_parametros_por_fold.csv")
comp_espec.to_csv(RES_DIR / "garch_comparacion_especificaciones.csv")
print("Archivos guardados en", RES_DIR)

## 10. Conclusiones del modelo econométrico
_Completar con los resultados:_
- Parámetros estimados y su interpretación (persistencia α + β, vida media de los choques, ν).
- Cumplimiento de supuestos: residuos sin autocorrelación ni efectos ARCH remanentes, ajuste de la distribución t, resultado del sign bias test.
- Comparación con especificaciones alternativas (AIC/BIC).
- Desempeño en los folds: en qué periodos el error es mayor (por ejemplo, 2020) y si los parámetros son estables al expandir el entrenamiento.
- Estos pronósticos (`garch_pronosticos_diarios.csv`) se comparan con los del Random Forest en el notebook de comparación.